# 01 Power Analysis and Experiment Design

Everything in this notebook was written before any outcome data was touched.
That ordering is the whole point of the notebook.

Two experiments run through this project.

The Criteo incrementality test is real, from a live advertising platform, and it
has already finished. That means the design phase for it happened years ago and
we cannot rerun it. What we can do is reconstruct what the design phase should
have concluded, and check whether the sample they collected was adequate.

The fintech onboarding test is simulated, precisely so that we can run the design
phase honestly on a problem where no data exists yet.

import sys
sys.path.append("..")

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.power import (
    required_sample_size,
    sample_size_curve,
    runtime_days,
    minimum_detectable_effect,
)

with open("../config/experiment_config.yaml") as f:
    config = yaml.safe_load(f)

config["design"]

import sys
sys.path.append("..")

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.power import (
    required_sample_size,
    sample_size_curve,
    runtime_days,
    minimum_detectable_effect,
)

with open("../config/experiment_config.yaml") as f:
    config = yaml.safe_load(f)

config["design"]

## Part one: designing the fintech onboarding test

The minimum detectable effect is a business decision, not a statistical one.
The question is not "what lift do we hope for" but "what is the smallest lift
that would justify the cost of building and maintaining this change".

Product agreed that a five percent relative lift in funded accounts is the floor.
Below that, the engineering cost is not recovered. We set the MDE slightly above
the floor, at six percent, so that if we detect an effect at all it is one we can
act on.

Baseline funding rate from the last ninety days of production data: 22 percent.

BASELINE = 0.22
MDE = 0.06
ALPHA = 0.05
POWER = 0.80

design = required_sample_size(BASELINE, MDE, alpha=ALPHA, power=POWER, ratio=1.0)

for k, v in design.items():
    print(f"{k:26s} {v}")

In [ ]:
DAILY_SIGNUPS = 2_800

days = runtime_days(design["n_total"], DAILY_SIGNUPS)
weeks = np.ceil(days / 7)

print(f"At {DAILY_SIGNUPS:,} signups a day, the test needs {days:.0f} days.")
print()
print(f"We round up to {weeks * 7:.0f} days, a whole number of weeks.")
print("This is not fussiness. Signup behaviour differs between weekdays and")
print("weekends, and a test that runs for ten days weights one more heavily than")
print("the other. Running whole weeks removes that source of bias for free.")

mdes = np.arange(0.02, 0.21, 0.005)
curve = pd.DataFrame(sample_size_curve(BASELINE, mdes))
curve["days"] = curve["n_total"] / DAILY_SIGNUPS

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(curve["relative_mde"] * 100, curve["n_total"] / 1000,
        linewidth=2.5, color="#1f4e79")
ax.axvline(MDE * 100, linestyle="--", color="#c0504d", linewidth=1.5)

ax.annotate(
    f"Our MDE: {MDE:.0%}\n{design['n_total']:,} users\n{days:.0f} days",
    xy=(MDE * 100, design["n_total"] / 1000),
    xytext=(MDE * 100 + 3.5, design["n_total"] / 1000 + 60),
    arrowprops=dict(arrowstyle="->", color="#c0504d"),
    fontsize=10,
)

two_pct = curve[curve["relative_mde"].round(3) == 0.020].iloc[0]
ax.annotate(
    f"Detecting 2%:\n{two_pct['n_total']:,.0f} users\n{two_pct['days']:.0f} days",
    xy=(2, two_pct["n_total"] / 1000),
    xytext=(4.5, two_pct["n_total"] / 1000 - 120),
    arrowprops=dict(arrowstyle="->", color="#7f7f7f"),
    fontsize=9, color="#555555",
)

ax.set_xlabel("Minimum detectable effect (relative, percent)")
ax.set_ylabel("Total users required (thousands)")
ax.set_title("The cost of chasing small effects")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/figures/01_sample_size_curve.png", dpi=150)
plt.show()

## What that chart is actually for

Take it into the planning meeting. When someone asks whether we could also detect
a two percent lift while we are at it, the answer is on the y axis: roughly nine
times the users, and a test that runs past the end of the quarter.

That is not a statistics objection, it is a prioritisation conversation, and the
chart has it for you without anyone needing to argue.

## Part two: was the Criteo experiment adequately powered

Criteo's conversion rate is 0.29 percent. Rare events are brutal for sample size,
because the variance of a proportion is p times one minus p, and when p is tiny
you need an enormous number of observations before the signal separates from the
noise.

This is the reason the dataset has fourteen million rows. It is not showing off.
It is the minimum viable size for the question being asked.

CRITEO_BASELINE_CONVERSION = 0.00292
CRITEO_BASELINE_VISIT = 0.046992
CRITEO_RATIO = 0.85 / 0.15   # treatment to control

for mde in [0.05, 0.10, 0.20]:
    d = required_sample_size(
        CRITEO_BASELINE_CONVERSION, mde,
        alpha=0.05, power=0.80, ratio=CRITEO_RATIO,
    )
    print(f"MDE {mde:>5.0%}   control {d['n_control']:>12,}   "
          f"treatment {d['n_treatment']:>12,}   total {d['n_total']:>13,}")

print()
print("Compare with the visit metric, which is sixteen times more common:")
for mde in [0.05, 0.10]:
    d = required_sample_size(
        CRITEO_BASELINE_VISIT, mde,
        alpha=0.05, power=0.80, ratio=CRITEO_RATIO,
    )
    print(f"MDE {mde:>5.0%}   total {d['n_total']:>13,}")

CRITEO_BASELINE_CONVERSION = 0.00292
CRITEO_BASELINE_VISIT = 0.046992
CRITEO_RATIO = 0.85 / 0.15   # treatment to control

for mde in [0.05, 0.10, 0.20]:
    d = required_sample_size(
        CRITEO_BASELINE_CONVERSION, mde,
        alpha=0.05, power=0.80, ratio=CRITEO_RATIO,
    )
    print(f"MDE {mde:>5.0%}   control {d['n_control']:>12,}   "
          f"treatment {d['n_treatment']:>12,}   total {d['n_total']:>13,}")

print()
print("Compare with the visit metric, which is sixteen times more common:")
for mde in [0.05, 0.10]:
    d = required_sample_size(
        CRITEO_BASELINE_VISIT, mde,
        alpha=0.05, power=0.80, ratio=CRITEO_RATIO,
    )
    print(f"MDE {mde:>5.0%}   total {d['n_total']:>13,}")

In [ ]:
from src.load_criteo import load_counts

counts = load_counts()

mde_conversion = minimum_detectable_effect(
    n_control=counts["n_control"],
    n_treatment=counts["n_treatment"],
    baseline_rate=counts["conversion_rate_control"],
)
mde_visit = minimum_detectable_effect(
    n_control=counts["n_control"],
    n_treatment=counts["n_treatment"],
    baseline_rate=counts["visit_rate_control"],
)

print(f"With the traffic Criteo actually collected:")
print(f"  Control arm   {counts['n_control']:,}")
print(f"  Treatment arm {counts['n_treatment']:,}")
print()
print(f"  Smallest conversion lift detectable at 80 percent power: {mde_conversion:.2%}")
print(f"  Smallest visit lift detectable at 80 percent power:      {mde_visit:.2%}")